In [1]:
!pwd

/content


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install chemprop --no-deps
!pip install rdkit-pypi pandas pyarrow scikit-learn rdkit

In [2]:
!pip install chemprop
!pip install pandas pyarrow scikit-learn rdkit

In [5]:
import pandas as pd

df = pd.read_parquet("dataset_raw.parquet")

print("\n=== RAW ===")
print("rows:", len(df))
print("unique smiles:", df["smiles"].nunique())
print("tasks:", df["task"].nunique())

print("\nSPLIT:")
print(df["split"].value_counts())

print("\nLABEL:")
print(df["label"].value_counts())



df["split"] = df["split"].replace({"valid": "val"})

print("\nSPLIT AFTER FIX:")
print(df["split"].value_counts())


df_wide = df.pivot_table(
    index="smiles",
    columns="task",
    values="label",
    aggfunc="first"
)

split_df = df.groupby("smiles")["split"].agg(lambda x: x.value_counts().index[0])

df_wide["split"] = split_df
df_wide = df_wide.reset_index()

print("\n=== WIDE ===")
print("rows:", len(df_wide))
print("cols:", len(df_wide.columns))
print(df_wide["split"].value_counts())




df_wide.to_csv("chemprop_data.csv", index=False)

tasks = [c for c in df_wide.columns if c not in ["smiles","split"]]

print("tasks:", len(tasks))





=== RAW ===
rows: 65127
unique smiles: 40000
tasks: 13

SPLIT:
split
train    45617
test     13002
valid     6508
Name: count, dtype: int64

LABEL:
label
0.0    38724
1.0    26403
Name: count, dtype: int64

SPLIT AFTER FIX:
split
train    45617
test     13002
val       6508
Name: count, dtype: int64

=== WIDE ===
rows: 40000
cols: 15
split
train    28989
test      7416
val       3595
Name: count, dtype: int64
tasks: 13


In [9]:

targets = " ".join([f'"{t}"' for t in tasks])


!chemprop train \
  --data-path chemprop_data.csv \
  --task-type classification \
  --smiles-columns smiles \
  --target-columns {targets} \
  --splits-column split \
  --epochs 30 \
  --batch-size 128 \
  --dropout 0.1 \
  --message-hidden-dim 300 \
  --depth 3 \
  --output-dir chemprop_model

Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
                                                               train_loss_epoch:
Epoch 26/29 ━━╺━━━━━━━━━━━━━ 31/227 0:00:00 •        87.92it/s v_num: 2.000     
                                    0:00:03                    train_loss_step: 
                                                               0.308 val_loss:  
                                                               0.412            
                                                               train_loss_epoch:
Epoch 26/29 ━━╺━━━━━━━━━━━━━ 32/227 0:00:00 •        87.88it/s v_num: 2.000     
                                    0:00:03                    train_loss_step: 
                                                               0.366 val_loss:  
                                                               0.412            
                                                               train_loss_epoch:
Epoch 26/29 ━━╺━━━━━━━━━━━━━ 33/227 0:00:00 

In [10]:
!chemprop fingerprint \
  --test-path chemprop_data.csv \
  --model-path chemprop_model/model_0/checkpoints/best-epoch=29-val_loss=0.41.ckpt \
  --ffn-block-index -1 \
  --output embeddings.csv

2026-03-26T12:13:12 - INFO:chemprop.cli.main - Running in mode 'fingerprint' with args: {'smiles_columns': None, 'reaction_columns': None, 'no_header_row': False, 'num_workers': 0, 'batch_size': 64, 'accelerator': 'auto', 'devices': 'auto', 'rxn_mode': 'REAC_DIFF', 'multi_hot_atom_featurizer_mode': 'V2', 'keep_h': False, 'add_h': False, 'ignore_stereo': False, 'reorder_atoms': False, 'molecule_featurizers': None, 'descriptors_path': None, 'descriptors_columns': None, 'no_descriptor_scaling': False, 'no_atom_feature_scaling': False, 'no_atom_descriptor_scaling': False, 'no_bond_feature_scaling': False, 'no_bond_descriptor_scaling': False, 'atom_features_path': None, 'atom_descriptors_path': None, 'bond_features_path': None, 'bond_descriptors_path': None, 'constraints_path': None, 'constraints_to_targets': None, 'use_cuikmolmaker_featurization': False, 'test_path': PosixPath('chemprop_data.csv'), 'output': PosixPath('embeddings.csv'), 'model_paths': [PosixPath('chemprop_model/model_0/che

In [11]:
import pandas as pd

emb = pd.read_csv("embeddings_0.csv")

smiles = pd.read_csv("chemprop_data.csv")["smiles"]

emb["smiles"] = smiles

print(emb.head())


print("shape:", emb.shape)
print("dim:", emb.shape[1]-1)
print("unique smiles:", emb["smiles"].nunique())



df = pd.read_parquet("dataset_raw.parquet")
df["split"] = df["split"].replace({"valid":"val"})

df = df.merge(emb, on="smiles")

print("rows:", len(df))
print("cols:", len(df.columns))


print("unique smiles:", df["smiles"].nunique())
print("tasks:", df["task"].nunique())
print("splits:", df["split"].value_counts())




       fp_0      fp_1      fp_2      fp_3      fp_4      fp_5      fp_6  \
0  0.838213 -0.344061  0.366978 -0.331949 -0.074869  1.304330 -0.168218   
1 -0.334595 -0.379314 -0.133814 -0.323606  0.103194  0.091770  0.457618   
2  0.898967 -0.404759  0.558646 -0.323147 -0.202356  0.680719  0.064725   
3  0.231508 -0.281415 -0.047721 -0.191074  0.615741 -0.172015  0.590929   
4  0.513150 -0.321835  0.360060 -0.283267 -0.305287  0.692376  0.089422   

       fp_7      fp_8      fp_9  ...    fp_291    fp_292    fp_293    fp_294  \
0 -0.159733 -0.495210 -0.305594  ...  0.082286 -0.286687  0.580209  0.446170   
1 -0.112076 -0.608285 -0.281877  ...  0.025808  0.135197 -0.351907 -0.926706   
2 -0.142454 -0.651192 -0.425231  ...  0.216572 -0.051515  0.534476  0.779840   
3 -0.105830 -0.395839 -0.292779  ...  0.087554  0.706782  0.012482 -0.143323   
4 -0.137204 -0.497368 -0.309108  ...  0.143464 -0.239700  0.352646  0.633051   

     fp_295    fp_296    fp_297    fp_298    fp_299  \
0  0.386023  

In [12]:

import pandas as pd

# embeddings już masz jako emb
emb.to_parquet(
    "embeddings.parquet",
    engine="pyarrow",
    compression="snappy"
)

print("saved: embeddings.parquet")


df.to_parquet(
    "dataset_with_embeddings.parquet",
    engine="pyarrow",
    compression="snappy"
)

print("saved: dataset_with_embeddings.parquet")


emb_check = pd.read_parquet("embeddings.parquet")
df_check = pd.read_parquet("dataset_with_embeddings.parquet")

print("emb:", emb_check.shape)
print("df:", df_check.shape)


saved: embeddings.parquet
saved: dataset_with_embeddings.parquet
emb: (40000, 301)
df: (65127, 513)


In [13]:
emb_cols = [c for c in df.columns if c.startswith("fp_")]

assert len(emb_cols) == 300
assert len(df) == 65127
assert df["task"].nunique() == 13
assert set(df["split"].unique()) == {"train","val","test"}

print("FINAL DATA READY ✅")

FINAL DATA READY ✅


In [17]:
import pandas as pd
import numpy as np

emb = pd.read_parquet("embeddings.parquet")

print("\n=== BASIC ===")
print("shape:", emb.shape)

# ✔ kolumny
assert "smiles" in emb.columns

fp_cols = [c for c in emb.columns if c.startswith("fp_")]

print("n_fp:", len(fp_cols))
assert len(fp_cols) == 300

# ✔ unique smiles
print("unique smiles:", emb["smiles"].nunique())
assert emb["smiles"].nunique() == len(emb)

# ✔ brak NaN
print("NaN:", emb.isna().sum().sum())
assert emb.isna().sum().sum() == 0

# ✔ wartości
print("\n=== STATS ===")
print(emb[fp_cols].describe().T[["mean","std"]].head())

# ✔ czy embeddingi nie są stałe
var = emb[fp_cols].var().mean()
print("mean variance:", var)
assert var > 1e-6

# ✔ duplikaty embeddingów
dup = emb.duplicated(subset=fp_cols).sum()
print("duplicate embeddings:", dup)


# wartosci ekstremalne embeddingów
print("\n=== RANGE ===")
print("min:", emb[fp_cols].min().min())
print("max:", emb[fp_cols].max().max())


# similarity
cos_sim = np.dot(
    emb[fp_cols].iloc[0],
    emb[fp_cols].iloc[1]
) / (
    np.linalg.norm(emb[fp_cols].iloc[0]) *
    np.linalg.norm(emb[fp_cols].iloc[1])
)

print("cosine similarity (sample):", cos_sim)



=== BASIC ===
shape: (40000, 301)
n_fp: 300
unique smiles: 40000
NaN: 0

=== STATS ===
          mean       std
fp_0  0.437221  0.595873
fp_1 -0.411372  0.159128
fp_2  0.286763  0.355388
fp_3 -0.354631  0.128724
fp_4  0.101976  0.677039
mean variance: 0.1398658255248413
duplicate embeddings: 2

=== RANGE ===
min: -26.895935
max: 14.9650135
cosine similarity (sample): -0.14377226341591076


In [15]:
df = pd.read_parquet("dataset_with_embeddings.parquet")

print("\n=== BASIC ===")
print("shape:", df.shape)

# ✔ kolumny
required = ["smiles","task","label","split"]
for c in required:
    assert c in df.columns

# ✔ split
print("\nSPLIT:")
print(df["split"].value_counts())
assert set(df["split"].unique()) == {"train","val","test"}

# ✔ label
print("\nLABEL:")
print(df["label"].value_counts())
assert set(df["label"].unique()) <= {0,1}

# ✔ task
print("\nTASKS:", df["task"].nunique())

# ✔ brak NaN
print("NaN:", df.isna().sum().sum())
assert df.isna().sum().sum() == 0













fp_cols = [c for c in df.columns if c.startswith("fp_")]

print("\n=== EMBEDDING ===")
print("n_fp:", len(fp_cols))
assert len(fp_cols) == 300

# ✔ embedding consistency (NAJWAŻNIEJSZE)
g = df.groupby("smiles")[fp_cols].nunique().max().max()
print("max unique embedding per smiles:", g)
assert g == 1
















rdkit_cols = [c for c in df.columns if c.startswith(("Mol","Chi","PEOE","fr_")) or "VSA" in c]

qc_cols = ["dipole","homo_lumo","electrons","energy"]
mask_cols = ["mask_dipole","mask_homo_lumo","mask_electrons","mask_energy"]

print("\n=== FEATURES ===")
print("emb:", len(fp_cols))
print("rdkit:", len(rdkit_cols))
print("qc:", len(qc_cols))
print("mask:", len(mask_cols))

total = len(fp_cols)+len(rdkit_cols)+4+4
print("TOTAL:", total)










# mask logic
for f,m in zip(qc_cols, mask_cols):
    bad = ((df[m]==0) & (df[f]!=0)).sum()
    print(f"{f} mask violations:", bad)

# RDKit success
if "success" in df.columns:
    bad = ((df["success"]==0) & (df[rdkit_cols].sum(axis=1)!=0)).sum()
    print("RDKit violations:", bad)








print("\n=== TASK DISTRIBUTION ===")
print(df.groupby("task")["label"].mean().sort_values())








split_per_smiles = df.groupby("smiles")["split"].nunique()

leak = (split_per_smiles > 1).sum()
print("SMILES in multiple splits:", leak)



print("\nALL CHECKS DONE ✅")




=== BASIC ===
shape: (65127, 513)

SPLIT:
split
train    45617
test     13002
val       6508
Name: count, dtype: int64

LABEL:
label
0.0    38724
1.0    26403
Name: count, dtype: int64

TASKS: 13
NaN: 0

=== EMBEDDING ===
n_fp: 300
max unique embedding per smiles: 1

=== FEATURES ===
emb: 300
rdkit: 157
qc: 4
mask: 4
TOTAL: 465
dipole mask violations: 0
homo_lumo mask violations: 0
electrons mask violations: 0
energy mask violations: 0
RDKit violations: 0

=== TASK DISTRIBUTION ===
task
cyp2d6_veith                      0.191470
cyp2c9_substrate_carbonmangels    0.211712
cyp2d6_substrate_carbonmangels    0.287651
cyp2c9_veith                      0.334519
cyp3a4_veith                      0.414504
dili                              0.496842
hERG_Karim                        0.499665
cyp3a4_substrate_carbonmangels    0.530735
pgp_broccatelli                   0.533828
ames                              0.544590
bbb_martins                       0.761519
bioavailability_ma                